![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)







# Import in apple verion
### Check Critical Package Version
✅ JAX: 0.6.2
✅ MuJoCo: 3.3.6
✅ Brax: 0.13.0
✅ Flax: 0.10.7

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from learning.notebooks.apple_mujoco_setup import *

Failed to import warp: No module named 'warp'
Failed to import mujoco.mjx.third_party.mujoco_warp as mujoco_warp: No module named 'warp'
Detected macOS: arm64
Using MUJOCO_GL=glfw for macOS
Forcing JAX to run on CPU backend
Mujoco installation and rendering backend OK
JAX: 0.6.2
MuJoCo: 3.3.6
Brax: 0.13.0
Flax: 0.10.7

Checking media packages...
✓ ffmpeg available
✓ mediapy available
[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7)]
JAX device count: 8


# Load Ur10PickCube environment

1. Create your env (same env + wrappers as training!)
2. Load checkpoint/params
3. Rebuild the exact same policy network apply_fn
4. Rollout

In [4]:
env_name = 'UR10PickCube'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)
m = env.mj_model



✓ Using keyframe: 'low_home'
  Initial qpos size: 15
  Robot joints: [ 0.   -1.26  1.57 -1.95 -1.5  -1.5   0.    0.  ]


In [ ]:
import jax
import jax.numpy as jnp

from trained_policy import (
    load_orbax_checkpoint,
    make_policy_from_apply_fn,
    jit_policy,
    rollout,
)

# 1) Build env (example; replace with your UR10 env builder)
# env = make_ur10_env(...)

# 2) Load checkpoint (orbax directory)
ckpt = load_orbax_checkpoint("/path/to/checkpoint_dir")

# Common patterns:
# params = ckpt["params"] or ckpt["policy_params"]
# obs_norm = ckpt.get("obs_norm") or ckpt.get("normalizer")
params = ckpt["params"]
obs_norm = ckpt.get("obs_norm", None)

# 3) Rebuild the policy apply_fn (YOU must replace this)
# For example, if you used a Flax module:
# action = policy_module.apply({"params": params}, obs)
def apply_fn(p, obs):
    # TODO: replace with your network forward pass
    # This stub assumes params is already "callable", which is not true for Flax.
    raise NotImplementedError("Replace apply_fn with your policy network forward pass.")

policy = make_policy_from_apply_fn(apply_fn=apply_fn, params=params, obs_normalizer=obs_norm)
policy = jit_policy(policy)

# 4) Rollout
traj = rollout(env, policy, seed=0, horizon=1000, deterministic=True)
print("Return:", traj["return"])
